# Turkey Fleet — Gaussian Copula Synthesizer

A faster, more interpretable alternative to CTGAN v4.

| | CTGAN v4 | **Gaussian Copula** |
|---|---|---|
| Training time | 38 min | < 1 min |
| Marginals | Post-hoc reweighting required | Exact — fit directly on Turkey |
| Dependency structure | Neural network (black box) | Correlation matrix Σ (interpretable) |
| FixedCombinations | Enforced | Filtered post-generation |

**Key idea**: train on Turkey rows only (6,845 rows) → marginals are Turkey-exact by construction,
no reweighting needed. Dependency between variables (energy ↔ segment, maker ↔ body_style)
is captured parametrically via a Gaussian correlation matrix.

## 1) Imports + data load

In [ ]:
try:
    import pandas as _pd
    _pd.options.future.infer_string = False
except AttributeError:
    pass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

train = pd.read_csv("../data/EM_LYON_train_set_20260206.csv", sep=";")
test  = pd.read_csv("../data/EM_LYON_test_set_20260206.csv")

train["country_iso"] = train["country_iso"].astype(str).str.strip().str.upper()
test["country_iso"]  = test["country_iso"].astype(str).str.strip().str.upper()

cat_cols = ["car_maker_name", "car_segment_name", "energy", "code_age", "body_style"]
train = train.dropna(subset=cat_cols).copy()

age_map = {
    "Less than 1 year old": 0.5, "1 year old": 1.0, "2 years old": 2.0,
    "3 to 5 years old": 4.0, "6 to 10 years old": 8.0, "11 years and older": 12.0,
}
train["age_years"] = train["code_age"].map(age_map)
test["age_years"]  = test["code_age"].map(age_map)

country_totals = train.groupby("country_iso")["total_vehicles"].sum()
train["log_total_vehicles"] = np.log1p(train["total_vehicles"])

turkey_total = float(country_totals.loc["TR"])
tr_real = train[train["country_iso"] == "TR"].copy()

# Valid (maker, segment, body_style) pairs from Turkey training data
valid_pairs = tr_real[["car_maker_name", "car_segment_name", "body_style"]].drop_duplicates()
valid_idx   = valid_pairs.set_index(["car_maker_name", "car_segment_name", "body_style"]).index

print(f"Turkey rows : {len(tr_real)}")
print(f"Turkey fleet: {turkey_total:,.0f}")
print(f"Valid (maker, segment, body_style) pairs: {len(valid_pairs)}")

## 2) Prepare Turkey-only training data

Unlike CTGAN which trains on all 11 countries, the Gaussian Copula trains on **Turkey rows only**.
This means the learned marginals are Turkey-exact — no post-hoc reweighting needed.

In [ ]:
cat_features = ["car_maker_name", "car_segment_name", "energy", "body_style", "code_age"]
num_features = ["age_years", "log_total_vehicles"]
config_cols  = ["car_maker_name", "car_segment_name", "energy", "body_style", "code_age"]
fb1_cols     = ["car_maker_name", "energy", "body_style"]
fb2_cols     = ["car_maker_name", "energy"]
fb3_cols     = ["energy"]

tr_data = tr_real[cat_features + num_features].copy()
for col in tr_data.select_dtypes(include="string").columns:
    tr_data[col] = tr_data[col].astype(object)

print(f"Training data shape : {tr_data.shape}")
print(f"Categorical features: {cat_features}")
print(f"Numerical features  : {num_features}")

## 3) Fit GaussianCopulaSynthesizer

The Gaussian Copula:
1. Fits marginal distributions to each column (categorical → frequency table, numerical → parametric)
2. Transforms all columns to standard normal via probability integral transform
3. Fits a correlation matrix **Σ** on the transformed space
4. At sampling time: draws from N(0, Σ), then inverts the transforms

In [ ]:
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(tr_data)
for c in cat_features: metadata.update_column(column_name=c, sdtype="categorical")
for c in num_features:  metadata.update_column(column_name=c, sdtype="numerical")

gc = GaussianCopulaSynthesizer(metadata)
gc.fit(tr_data)
print("GaussianCopulaSynthesizer fitted.")

# Inspect learned distributions per column
try:
    learned = gc.get_learned_distributions()
    print("\nLearned distributions:")
    for col, info in learned.items():
        print(f"  {col:<26}: {info}")
except Exception as e:
    print(f"(get_learned_distributions not available: {e})")

## 4) Sample 100k rows + filter invalid pairs\n",
    "\n",
    "No `Condition()` needed — the model was trained on Turkey rows only, so all sampled rows\n",
    "represent Turkey fleet compositions.\n",
    "\n",
    "**Invalid pair filtering**: Gaussian Copula does not enforce `FixedCombinations`, so some\n",
    "(maker, segment, body_style) triples may not exist in Turkey. We filter these out."

In [ ]:
N_SAMPLES = 200_000   # oversample to account for filtering
gc_raw = gc.sample(num_rows=N_SAMPLES)

# Filter invalid (maker, segment, body_style) pairs
gc_idx = gc_raw.set_index(["car_maker_name", "car_segment_name", "body_style"]).index
mask_valid = gc_idx.isin(valid_idx)
gc_synth = gc_raw[mask_valid].reset_index(drop=True)

pct_invalid = (~mask_valid).mean() * 100
print(f"Sampled       : {len(gc_raw):,}")
print(f"Invalid pairs : {(~mask_valid).sum():,} ({pct_invalid:.1f}%)")
print(f"Valid synth   : {len(gc_synth):,}")

if len(gc_synth) < 50_000:
    print("WARNING: fewer than 50k valid rows — consider increasing N_SAMPLES")

## 4b) Vehicle-weighted importance reweighting\n\nThe Gaussian Copula learned row-count marginals (each config row weighted equally).\nBut a config with 50,000 vehicles should drive the distribution 500× more than one with 100.\nWe apply the same vehicle-weighted importance sampling as CTGAN v4 to correct this.

In [ ]:
REWEIGHT_COLS = ["car_segment_name", "energy", "code_age"]
IW_CAP        = 30
N_RESAMPLE    = 100_000

# Vehicle-weighted real Turkey marginals
real_marginals = {}
for col in REWEIGHT_COLS:
    w = tr_real.groupby(col)["total_vehicles"].sum()
    real_marginals[col] = w / w.sum()
    print(f"\n{col} (vehicle-weighted):")
    print(real_marginals[col].sort_values(ascending=False).to_string())

def compute_iw(synth_df, real_marginals, cap=IW_CAP):
    df = synth_df.copy()
    for col, real_dist in real_marginals.items():
        synth_dist = df[col].value_counts(normalize=True)
        p_real  = df[col].map(real_dist).fillna(0)
        p_synth = df[col].map(synth_dist).fillna(1e-9)
        df[f"iw_{col}"] = (p_real / p_synth).clip(upper=cap)
    iw_cols = [f"iw_{c}" for c in real_marginals]
    df["iw_combined"] = df[iw_cols].product(axis=1).clip(upper=cap)
    return df

gc_iw = compute_iw(gc_synth, real_marginals)

# ESS diagnostic
w = gc_iw["iw_combined"]
ess = w.sum()**2 / (w**2).sum()
print(f"\nGC  iw_combined — mean={w.mean():.3f}  max={w.max():.1f}  "
      f"capped={(w >= IW_CAP).mean()*100:.1f}%  ESS={ess:,.0f}/{len(gc_iw):,}")

gc_rw = (
    gc_iw.sample(n=N_RESAMPLE, replace=True, weights="iw_combined", random_state=42)
    .drop(columns=[c for c in gc_iw.columns if c.startswith("iw_")])
    .reset_index(drop=True)
)
print(f"\nReweighted GC: {gc_rw.shape}")

## 5) TVD comparison — Gaussian Copula vs CTGAN v4

In [ ]:
def tvd_discrete(real_s, synth_s, real_weights=None):
    cats = set(real_s.dropna().unique()) | set(synth_s.dropna().unique())
    if real_weights is not None:
        real_p = (real_weights / real_weights.sum()).to_dict()
    else:
        real_p = real_s.value_counts(normalize=True).to_dict()
    synth_p = synth_s.value_counts(normalize=True).to_dict()
    return 0.5 * sum(abs(real_p.get(c, 0.0) - synth_p.get(c, 0.0)) for c in cats)

def tvd_continuous(real_s, synth_s):
    n    = real_s.dropna().shape[0]
    bins = int(np.ceil(np.log2(n) + 1))   # Sturges' rule
    combined = pd.concat([real_s, synth_s]).dropna()
    edges  = np.linspace(combined.min(), combined.max(), bins + 1)
    r_hist = np.histogram(real_s.dropna(),  bins=edges)[0].astype(float)
    s_hist = np.histogram(synth_s.dropna(), bins=edges)[0].astype(float)
    r_p = r_hist / (r_hist.sum() + 1e-12)
    s_p = s_hist / (s_hist.sum() + 1e-12)
    return 0.5 * np.sum(np.abs(r_p - s_p))

eval_cat  = ["car_maker_name", "car_segment_name", "energy", "body_style", "code_age"]
eval_cont = ["age_years"]

# Vehicle-weighted real Turkey marginals
vw = {col: tr_real.groupby(col)["total_vehicles"].sum() for col in eval_cat}

# Load CTGAN v4 for comparison (if available)
import os
ctgan_v4 = pd.read_csv("../data/ctgan_synth_turkey_v4.csv") \
    if os.path.exists("../data/ctgan_synth_turkey_v4.csv") else None

print(f"{'Column':<26} {'GC (raw)':>10} {'GC (rw)':>10}  {'CTGAN v4':>10}  Flag")
print("-" * 72)
for col in eval_cat:
    w    = vw.get(col)
    raw  = tvd_discrete(tr_real[col], gc_synth[col], w)
    rw   = tvd_discrete(tr_real[col], gc_rw[col],    w)
    ct   = tvd_discrete(tr_real[col], ctgan_v4[col], w) if ctgan_v4 is not None else float("nan")
    flag = "<-- HIGH" if rw > 0.1 else ""
    print(f"{col:<26} {raw:>10.4f} {rw:>10.4f}  {ct:>10.4f}  {flag}")

for col in eval_cont:
    if col in gc_rw.columns and col in tr_real.columns:
        raw  = tvd_continuous(tr_real[col], gc_synth[col])
        rw   = tvd_continuous(tr_real[col], gc_rw[col])
        ct   = tvd_continuous(tr_real[col], ctgan_v4[col]) if ctgan_v4 is not None else float("nan")
        flag = "<-- HIGH" if rw > 0.1 else ""
        print(f"{col:<26} {raw:>10.4f} {rw:>10.4f}  {ct:>10.4f}  {flag}")

# Bar chart — raw vs reweighted
all_cols = eval_cat + [c for c in eval_cont if c in gc_rw.columns and c in tr_real.columns]
gc_raw_vals = [tvd_discrete(tr_real[c], gc_synth[c], vw.get(c)) if c in eval_cat
               else tvd_continuous(tr_real[c], gc_synth[c]) for c in all_cols]
gc_rw_vals  = [tvd_discrete(tr_real[c], gc_rw[c],    vw.get(c)) if c in eval_cat
               else tvd_continuous(tr_real[c], gc_rw[c])    for c in all_cols]

x = range(len(all_cols))
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar([i - 0.2 for i in x], gc_raw_vals, 0.4, label="GC raw",        alpha=0.65, color="steelblue")
ax.bar([i + 0.2 for i in x], gc_rw_vals,  0.4, label="GC reweighted", alpha=0.85, color="forestgreen")
ax.axhline(0.1, color="red", linestyle="--", linewidth=1.2, label="TVD = 0.1 threshold")
ax.set_xticks(list(x))
ax.set_xticklabels(all_cols, rotation=20, ha="right")
ax.set_ylabel("TVD (vehicle-weighted)")
ax.set_title("Gaussian Copula: raw vs reweighted TVD")
ax.legend()
plt.tight_layout()
plt.show()


## 6) Predict Turkey test rows

In [ ]:
test_tr = test[test["country_iso"] == "TR"].copy()
print(f"Turkey test rows: {len(test_tr)}")

def synth_to_share(synth_df, col_name):
    g = synth_df.groupby(config_cols).size().reset_index(name="cnt")
    g[col_name] = g["cnt"] / len(synth_df)
    return g[config_cols + [col_name]]

def get_fallback(synth_df, cols, col_name):
    g = synth_df.groupby(cols).size().reset_index(name="cnt")
    g[col_name] = g["cnt"] / len(synth_df)
    return g[cols + [col_name]]

# Use reweighted GC data for prediction
sim = synth_to_share(gc_rw, "sim_share")
fb1 = get_fallback(gc_rw, fb1_cols, "fb1")
fb2 = get_fallback(gc_rw, fb2_cols, "fb2")
fb3 = get_fallback(gc_rw, fb3_cols, "fb3")
fb4 = float(sim["sim_share"].mean())

t = test_tr.copy()
t = t.merge(sim, on=config_cols, how="left")
t = t.merge(fb1, on=fb1_cols,   how="left")
t = t.merge(fb2, on=fb2_cols,   how="left")
t = t.merge(fb3, on=fb3_cols,   how="left")

t["pred_share"] = (
    t["sim_share"]
    .fillna(t["fb1"])
    .fillna(t["fb2"])
    .fillna(t["fb3"])
    .fillna(fb4)
)
t["pred_total_vehicles"] = t["pred_share"] * turkey_total

assert t["pred_share"].isna().sum() == 0, "NaN in pred_share"
n_sim = t["sim_share"].notna().sum()
print(f"Direct sim coverage: {n_sim}/{len(t)} rows")
print(f"Pred sum           : {t['pred_total_vehicles'].sum():,.0f}")
print(f"Turkey fleet total : {turkey_total:,.0f}")

## 7) Save synthetic data + predictions

In [ ]:
gc_synth.to_csv("../data/gc_synth_turkey.csv", index=False)
t[["pred_total_vehicles"]].to_csv("../data/turkey_predictions_gc.csv", index=False)

print(f"Saved gc_synth_turkey.csv        ({len(gc_synth):,} rows)")
print(f"Saved turkey_predictions_gc.csv  ({len(t)} rows)")